Add current directory

In [69]:
import sys
import os

# Add the datamodule directory to Python path
datamodule_dir = '/workspace/emp/src/datamodule'
if datamodule_dir not in sys.path:
    sys.path.insert(0, datamodule_dir)

print(f"Added to Python path: {datamodule_dir}")
print(f"Directory exists: {os.path.exists(datamodule_dir)}")

Added to Python path: /workspace/emp/src/datamodule
Directory exists: True


In [70]:
import traceback
from pathlib import Path
from typing import List

import av2.geometry.interpolate as interp_utils
import numpy as np
import torch
from av2.map.map_api import ArgoverseStaticMap

from av2_data_utils import (
    OBJECT_TYPE_MAP,
    OBJECT_TYPE_MAP_COMBINED,
    LaneTypeMap,
    load_av2_df,
)

In [71]:
raw_path = Path("/raid/datasets/av2/val/00010486-9a07-48ae-b493-cf4545855937") / "scenario_00010486-9a07-48ae-b493-cf4545855937.parquet"
df, am, scenario_id = load_av2_df(raw_path)
print(df.shape)
df.head()

(3152, 16)


,observed,track_id,object_type,object_category,timestep,position_x,position_y,heading,velocity_x,velocity_y,scenario_id,start_timestamp,end_timestamp,num_timestamps,focal_track_id,city
0,True,77543,vehicle,1,0,-11.823674,-567.402397,2.850212,-10.135780,3.035927,00010486-9a07-48ae-b493-cf4545855937,3.159773e+17,3.159774e+17,110,77544,austin
1,True,77543,vehicle,1,1,-12.393206,-567.225316,2.850076,-10.164790,3.032990,00010486-9a07-48ae-b493-cf4545855937,3.159773e+17,3.159774e+17,110,77544,austin
2,True,77543,vehicle,1,2,-13.080372,-567.009694,2.849730,-10.185672,3.031219,00010486-9a07-48ae-b493-cf4545855937,3.159773e+17,3.159774e+17,110,77544,austin
3,True,77543,vehicle,1,3,-13.880497,-566.754410,2.849045,-10.144089,3.014031,00010486-9a07-48ae-b493-cf4545855937,3.159773e+17,3.159774e+17,110,77544,austin
4,True,77543,vehicle,1,4,-14.773627,-566.465386,2.847962,-10.107290,2.997265,00010486-9a07-48ae-b493-cf4545855937,3.159773e+17,3.159774e+17,110,77544,austin


In [72]:
agent_id = df["focal_track_id"].values[0]
local_df = df[df["track_id"] == agent_id].iloc
agent_id

'77544'

In [73]:
origin = torch.tensor(
            [local_df[49]["position_x"], local_df[49]["position_y"]], dtype=torch.float
        )
origin

tensor([ -11.6492, -567.4967])

In [74]:
theta = torch.tensor([local_df[49]["heading"]], dtype=torch.float)
theta

tensor([2.8417])

In [75]:
rotate_mat = torch.tensor(
            [
                [torch.cos(theta), -torch.sin(theta)],
                [torch.sin(theta), torch.cos(theta)],
            ],
        )
rotate_mat

tensor([[-0.9554, -0.2954],
        [ 0.2954, -0.9554]])

In [76]:
timestamps = list(np.sort(df["timestep"].unique()))


In [77]:
cur_df = df[df["timestep"] == timestamps[49]]
cur_df.head()

,observed,track_id,object_type,object_category,timestep,position_x,position_y,heading,velocity_x,velocity_y,scenario_id,start_timestamp,end_timestamp,num_timestamps,focal_track_id,city
49,True,77543,vehicle,1,49,-60.016199,-550.841971,2.802868,-10.368299,3.634337,00010486-9a07-48ae-b493-cf4545855937,3.159773e+17,3.159774e+17,110,77544,austin
159,True,77544,vehicle,3,49,-11.649155,-567.496714,2.841739,-8.294655,2.571242,00010486-9a07-48ae-b493-cf4545855937,3.159773e+17,3.159774e+17,110,77544,austin
269,True,77812,vehicle,0,49,74.049987,-609.847571,-2.335026,-1.380514,-1.571147,00010486-9a07-48ae-b493-cf4545855937,3.159773e+17,3.159774e+17,110,77544,austin
439,True,78008,vehicle,0,49,-84.070557,-542.751907,2.823819,-10.166234,3.366654,00010486-9a07-48ae-b493-cf4545855937,3.159773e+17,3.159774e+17,110,77544,austin
521,True,78019,motorcyclist,0,49,11.160624,-568.915890,2.782005,-1.438822,0.598081,00010486-9a07-48ae-b493-cf4545855937,3.159773e+17,3.159774e+17,110,77544,austin


In [78]:
actor_ids = list(cur_df["track_id"].unique())
len(actor_ids), actor_ids

(28,
 ['77543',
  '77544',
  '77812',
  '78008',
  '78019',
  '78037',
  '78055',
  '78058',
  '78076',
  '78096',
  '78097',
  '78100',
  '78102',
  '78103',
  '78104',
  '78106',
  '78110',
  '78111',
  '78113',
  '78115',
  '78117',
  '78118',
  '78119',
  '78120',
  '78123',
  '78124',
  '78128',
  'AV'])

In [79]:
pos_df = cur_df[["position_x", "position_y"]]#.values
pos_df.head()

,position_x,position_y
49,-60.016199,-550.841971
159,-11.649155,-567.496714
269,74.049987,-609.847571
439,-84.070557,-542.751907
521,11.160624,-568.915890


In [80]:
cur_pos = torch.from_numpy(cur_df[["position_x", "position_y"]].values).float()
cur_pos.shape

torch.Size([28, 2])

In [81]:
out_of_range = np.linalg.norm(cur_pos - origin, axis=1) > 150
out_of_range.shape, out_of_range.sum()

((28,), 0)

In [82]:
actor_ids = [aid for i, aid in enumerate(actor_ids) if not out_of_range[i]]
len(actor_ids), actor_ids

(28,
 ['77543',
  '77544',
  '77812',
  '78008',
  '78019',
  '78037',
  '78055',
  '78058',
  '78076',
  '78096',
  '78097',
  '78100',
  '78102',
  '78103',
  '78104',
  '78106',
  '78110',
  '78111',
  '78113',
  '78115',
  '78117',
  '78118',
  '78119',
  '78120',
  '78123',
  '78124',
  '78128',
  'AV'])

In [83]:
actor_ids.remove(agent_id)


In [84]:
actor_ids

['77543',
 '77812',
 '78008',
 '78019',
 '78037',
 '78055',
 '78058',
 '78076',
 '78096',
 '78097',
 '78100',
 '78102',
 '78103',
 '78104',
 '78106',
 '78110',
 '78111',
 '78113',
 '78115',
 '78117',
 '78118',
 '78119',
 '78120',
 '78123',
 '78124',
 '78128',
 'AV']

In [85]:
actor_ids = [agent_id] + actor_ids

In [86]:
actor_ids

['77544',
 '77543',
 '77812',
 '78008',
 '78019',
 '78037',
 '78055',
 '78058',
 '78076',
 '78096',
 '78097',
 '78100',
 '78102',
 '78103',
 '78104',
 '78106',
 '78110',
 '78111',
 '78113',
 '78115',
 '78117',
 '78118',
 '78119',
 '78120',
 '78123',
 '78124',
 '78128',
 'AV']

In [87]:
num_nodes = len(actor_ids)
num_nodes

28

In [88]:
df = df[df["track_id"].isin(actor_ids)]
df.shape

(2043, 16)

In [89]:
def get_lane_features(
        am: ArgoverseStaticMap,
        query_pos: torch.Tensor,
        origin: torch.Tensor,
        rotate_mat: torch.Tensor,
        radius: float,
    ):
        lane_segments = am.get_nearby_lane_segments(query_pos.numpy(), radius)

        lane_positions, is_intersections, lane_attrs = [], [], []
        for segment in lane_segments:
            lane_centerline, lane_width = interp_utils.compute_midpoint_line(
                left_ln_boundary=segment.left_lane_boundary.xyz,
                right_ln_boundary=segment.right_lane_boundary.xyz,
                num_interp_pts=20,
            )
            lane_centerline = torch.from_numpy(lane_centerline[:, :2]).float()
            lane_centerline = torch.matmul(lane_centerline - origin, rotate_mat)
            is_intersection = am.lane_is_in_intersection(segment.id)

            lane_positions.append(lane_centerline)
            is_intersections.append(is_intersection)

            # get lane attrs
            lane_type = LaneTypeMap[segment.lane_type]
            attribute = torch.tensor(
                [lane_type, lane_width, is_intersection], dtype=torch.float
            )
            lane_attrs.append(attribute)

        lane_positions = torch.stack(lane_positions)
        lanes_ctr = lane_positions[:, 9:11].mean(dim=1)
        lanes_angle = torch.atan2(
            lane_positions[:, 10, 1] - lane_positions[:, 9, 1],
            lane_positions[:, 10, 0] - lane_positions[:, 9, 0],
        )
        is_intersections = torch.Tensor(is_intersections)
        lane_attrs = torch.stack(lane_attrs, dim=0)

        x_max, x_min = radius, -radius
        y_max, y_min = radius, -radius

        padding_mask = (
            (lane_positions[:, :, 0] > x_max)
            | (lane_positions[:, :, 0] < x_min)
            | (lane_positions[:, :, 1] > y_max)
            | (lane_positions[:, :, 1] < y_min)
        )

        invalid_mask = padding_mask.all(dim=-1)
        lane_positions = lane_positions[~invalid_mask]
        is_intersections = is_intersections[~invalid_mask]
        lane_attrs = lane_attrs[~invalid_mask]
        lanes_ctr = lanes_ctr[~invalid_mask]
        lanes_angle = lanes_angle[~invalid_mask]
        padding_mask = padding_mask[~invalid_mask]

        lane_positions = torch.where(
            padding_mask[..., None], torch.zeros_like(lane_positions), lane_positions
        )

        return (
            lane_positions,
            is_intersections,
            lanes_ctr,
            lanes_angle,
            lane_attrs,
            padding_mask,
        )

In [90]:
# initialization
x = torch.zeros(num_nodes, 110, 2, dtype=torch.float)
ori = torch.zeros(num_nodes, 110, 2, dtype=torch.float)
ori_tf = torch.zeros(num_nodes, 110, 2, dtype=torch.float)
x_attr = torch.zeros(num_nodes, 3, dtype=torch.int)
x_heading = torch.zeros(num_nodes, 110, dtype=torch.float)
x_velocity = torch.zeros(num_nodes, 110, dtype=torch.float)
x_track_horizon = torch.zeros(num_nodes, dtype=torch.int)
padding_mask = torch.ones(num_nodes, 110, dtype=torch.bool)

In [91]:
mode = "train"
radius = 150.0

In [92]:
for actor_id, actor_df in df.groupby("track_id"):
    node_idx = actor_ids.index(actor_id)
    node_steps = [timestamps.index(ts) for ts in actor_df["timestep"]]
    object_type = OBJECT_TYPE_MAP[actor_df["object_type"].values[0]]
    x_attr[node_idx, 0] = object_type
    x_attr[node_idx, 1] = actor_df["object_category"].values[0]
    x_attr[node_idx, 2] = OBJECT_TYPE_MAP_COMBINED[
        actor_df["object_type"].values[0]
    ]
    x_track_horizon[node_idx] = node_steps[-1] - node_steps[0]
    padding_mask[node_idx, node_steps] = False
    if padding_mask[node_idx, 49] or object_type in [5, 6, 7, 8, 9]:
                padding_mask[node_idx, 50:] = True
    pos_xy = torch.from_numpy(
                np.stack(
                    [actor_df["position_x"].values, actor_df["position_y"].values],
                    axis=-1,
                )
            ).float()
    heading = torch.from_numpy(actor_df["heading"].values).float()
    velocity = torch.from_numpy(
                actor_df[["velocity_x", "velocity_y"]].values
            ).float()
    velocity_norm = torch.norm(velocity, dim=1)
    ori[node_idx, node_steps, :2] = pos_xy.clone()
    x[node_idx, node_steps, :2] = torch.matmul(pos_xy - origin, rotate_mat)
    # print(pos_xy.shape, x[node_idx, node_steps, :2].shape)
    # print(pos_xy[:3, :])
    # print(x[node_idx, node_steps[:3], :2])
    x_heading[node_idx, node_steps] = (heading - theta + np.pi) % (
        2 * np.pi
    ) - np.pi
    x_velocity[node_idx, node_steps] = velocity_norm
    # break
    # if node_idx > 1:
    #     break

(   lane_positions,
    is_intersections,
    lane_ctrs,
    lane_angles,
    lane_attr,
    lane_padding_mask,
    ) = get_lane_features(am, origin, origin, rotate_mat, radius)

print(x.shape)
if True:
    lane_samples = lane_positions[:, ::1, :2].view(-1, 2)
    nearest_dist = torch.cdist(x[:, 49, :2], lane_samples).min(dim=1).values
    valid_actor_mask = nearest_dist < 5
    valid_actor_mask[0] = True  # always keep the target agent

    x = x[valid_actor_mask]
    ori = ori[valid_actor_mask]
    x_heading = x_heading[valid_actor_mask]
    x_velocity = x_velocity[valid_actor_mask]
    x_attr = x_attr[valid_actor_mask]
    padding_mask = padding_mask[valid_actor_mask]
    num_nodes = x.shape[0]

print(x[3, 49:55, :])
x_ctrs = x[:, 49, :2].clone()
x_positions = x[:, :50, :2].clone()
x_velocity_diff = x_velocity[:, :50].clone()

x[:, 50:] = torch.where(
    (padding_mask[:, 49].unsqueeze(-1) | padding_mask[:, 50:]).unsqueeze(-1),
    torch.zeros(num_nodes, 60, 2),
    x[:, 50:] - x[:, 49].unsqueeze(-2),
)

present_ref = x[:, 49].clone()

x[:, 1:50] = torch.where(
    (padding_mask[:, :49] | padding_mask[:, 1:50]).unsqueeze(-1),
    torch.zeros(num_nodes, 49, 2),
    x[:, 1:50] - x[:, :49],
)
x[:, 0] = torch.zeros(num_nodes, 2)

x_velocity_diff[:, 1:50] = torch.where(
    (padding_mask[:, :49] | padding_mask[:, 1:50]),
    torch.zeros(num_nodes, 49),
    x_velocity_diff[:, 1:50] - x_velocity_diff[:, :49],
)
x_velocity_diff[:, 0] = torch.zeros(num_nodes)

y = None if mode == "test" else x[:, 50:]

# return {
#     "x": x[:, :50],
#     "y": y,
#     "x_attr": x_attr,
#     "x_positions": x_positions,
#     "x_centers": x_ctrs,
#     "x_angles": x_heading,
#     "x_velocity": x_velocity,
#     "x_velocity_diff": x_velocity_diff,
#     "x_padding_mask": padding_mask,
#     "lane_positions": lane_positions,
#     "lane_centers": lane_ctrs,
#     "lane_angles": lane_angles,
#     "lane_attr": lane_attr,
#     "lane_padding_mask": lane_padding_mask,
#     "is_intersections": is_intersections,
#     "origin": origin.view(-1, 2),
#     "theta": theta,
#     "scenario_id": scenario_id,
#     "track_id": agent_id,
#     "city": city,
# }


torch.Size([28, 110, 2])
tensor([[76.4991, -2.2488],
        [77.5496, -2.2569],
        [78.5319, -2.2648],
        [79.4291, -2.2726],
        [80.2291, -2.2804],
        [80.9250, -2.2883]])


In [93]:
x[3, 49:55, :]

tensor([[ 1.0967, -0.0082],
        [ 1.0505, -0.0080],
        [ 2.0329, -0.0160],
        [ 2.9300, -0.0238],
        [ 3.7300, -0.0315],
        [ 4.4259, -0.0395]])

In [94]:
x[3, 49:55, :] + present_ref[3, :].repeat(6, 1)

tensor([[77.5958, -2.2570],
        [77.5496, -2.2569],
        [78.5319, -2.2648],
        [79.4291, -2.2726],
        [80.2291, -2.2804],
        [80.9250, -2.2883]])

In [95]:
local_positions = x[3, 50:55, :] + present_ref[3, :].repeat(5, 1)

rotate_mat = torch.tensor([[-0.9554, -0.2954],
                           [ 0.2954, -0.9554]])

global_positions = torch.matmul(local_positions, rotate_mat.T) + origin

In [96]:
def predictions_to_global_coordinates_all_agents(
    predictions: torch.Tensor,   # [n_agents, n_modes, 60, 2]
    present_ref: torch.Tensor,   # [n_agents, 2]  (posición en t=49 en sistema local)
    rotate_mat: torch.Tensor,    # [2, 2]         (matriz de rotación del agente focal)
    origin: torch.Tensor         # [2]            (posición global del agente focal)
) -> torch.Tensor:
    """
    Transforma las trayectorias relativas predichas al sistema global.
    """
    n_agents, n_modes, n_steps, _ = predictions.shape

    # 1. Sumar posición en t=49 (referencia local)
    local_positions = predictions + present_ref[:, None, None, :]  # [N, M, 60, 2]

    # 2. Aplicar rotación inversa
    rotate_mat_inv = rotate_mat.T  # [2, 2]
    global_positions = torch.matmul(local_positions, rotate_mat_inv)  # [N, M, 60, 2]

    # 3. Trasladar al origen global del agente focal
    global_positions = global_positions + origin.view(1, 1, 1, 2)  # broadcast automático

    return global_positions

In [97]:
def predictions_to_global_coordinates_single_mode(
    predictions: torch.Tensor,   # [n_agents, 60, 2] → desplazamientos relativos
    present_ref: torch.Tensor,   # [n_agents, 2]     → posición en t=49 (local)
    theta: torch.Tensor,         # [1]               → theta del focal agent
    origin: torch.Tensor         # [2]               → posición global del focal agent
) -> torch.Tensor:
    """
    Convierte predicciones relativas al sistema global, para un único modo por agente.
    """
    # 1. Pasar a posiciones locales absolutas
    local_positions = predictions + present_ref.unsqueeze(1)  # [N, 60, 2]

    rotate_mat = torch.tensor(
        [
            [torch.cos(theta), -torch.sin(theta)],
            [torch.sin(theta), torch.cos(theta)],
        ],
    )

    # 2. Rotar hacia el sistema global (R^T)
    global_positions = torch.matmul(local_positions, rotate_mat.T)  # [N, 60, 2]

    # 3. Trasladar al origen global del focal agent
    global_positions = global_positions + origin.view(1, 1, 2)  # broadcasting automático

    return global_positions

In [98]:
padding_mask[:, 50:].shape

torch.Size([27, 60])

In [99]:
theta

tensor([2.8417])

In [106]:
# global_positions = predictions_to_global_coordinates_single_mode(
#     y,
#     present_ref,
#     rotate_mat,
#     origin
# )

# global_positions[3, :5, :]



global_positions = predictions_to_global_coordinates_single_mode(
    y,
    present_ref,
    theta,  # Now passing scalar angle
    origin
)

global_positions[3, :5, :]

tensor([[ -85.0715, -542.4330],
        [ -86.0077, -542.1352],
        [ -86.8625, -541.8627],
        [ -87.6245, -541.6190],
        [ -88.2870, -541.4059]])

In [107]:
ori[3, 50:55, :]

tensor([[ -85.0718, -542.4339],
        [ -86.0080, -542.1362],
        [ -86.8628, -541.8637],
        [ -87.6248, -541.6200],
        [ -88.2874, -541.4069]])

In [108]:
global_positions.shape

torch.Size([27, 60, 2])

In [109]:
ori.shape

torch.Size([27, 110, 2])

In [110]:
global_positions_mask = global_positions.masked_fill(padding_mask[:, 50:].unsqueeze(-1), 0.0)

In [111]:
import torch

# Diferencia absoluta
diff = global_positions_mask - ori[:, 50:, :2]  # (B, A, 60, 2)

# Distancia euclidiana por punto
error_per_point = torch.norm(diff, dim=-1)  # (B, A, 60)

# Métricas de evaluación
max_error = error_per_point.max().item()
mean_error = error_per_point.mean().item()

print(f"Máximo error: {max_error:.6f} m")
print(f"Error medio: {mean_error:.6f} m")

# Opcional: umbral para considerar igual (e.g. 1e-4 m)
threshold = 0.1
num_close = (error_per_point < threshold).sum()
total_points = error_per_point.numel()
print(f"Porcentaje de puntos con error < {threshold}: {100 * num_close / total_points:.2f}%")

Máximo error: 582.962341 m
Error medio: 225.684509 m
Porcentaje de puntos con error < 0.1: 60.68%
